# 22AICP409 — Machine Learning Laboratory Manual (Complete Codebook)
**Annamalai University — Faculty of Engineering and Technology**  
**Department of Computer Science and Engineering (AI & ML)**  
**Semester:** IV | **Course Code:** 22AICP409

## `ex01a_linear_regression.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Load dataset
data = pd.read_csv("../MLdatasets/headbrain.csv")
print(data)

X = data["Head Size(cm^3)"]
y = data["Brain Weight(grams)"]

# Split into training and testing sets (33% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Train Linear Regression model
model = LinearRegression()
model.fit(np.expand_dims(X_train, axis=1), np.expand_dims(y_train, axis=1))
predictions = model.predict(np.expand_dims(X_test, axis=1))

# Plot scatter points and regression line
plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, color="r", label='Scatter Plot')
plt.plot(X_test, predictions, color="b", label='Regression Line')
plt.xlabel('Head Size in cm3')
plt.ylabel('Brain Weight in grams')
plt.legend()
plt.show()

# Error Estimation (RMSE & R2 Score)
r2_score = model.score(np.expand_dims(X_test, axis=1), np.expand_dims(y_test, axis=1))
mse = mean_squared_error(y_test, predictions)
print(f"RMSE VALUE : {np.sqrt(mse)}\nR2 SCORE : {r2_score}")


## `ex01b_logistic_titanic.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Load dataset and visualize missing values
titanic_data = pd.read_csv("../MLdatasets/titanic.csv")
sns.heatmap(titanic_data.isnull(), cbar=False)
plt.show()

sns.countplot(x='Survived', hue='Sex', data=titanic_data)
plt.show()

sns.countplot(x='Survived', hue='Pclass', data=titanic_data)
plt.show()

# Fill null values of column [Age] by mean age of people in the same Pclass
def get_age(row):
    if pd.isnull(row["Age"]):
        same_class = titanic_data[titanic_data["Pclass"] == row["Pclass"]]
        return same_class["Age"].mean()
    else:
        return row["Age"]

titanic_data["Age"] = titanic_data[["Age", "Pclass"]].apply(get_age, axis=1)
titanic_data.drop("Cabin", axis=1, inplace=True)
titanic_data.dropna(inplace=True)

sns.heatmap(titanic_data.isnull(), cbar=False)
plt.show()

# One-hot encode categorical columns
sex_data = pd.get_dummies(titanic_data["Sex"]).astype(int)
embarked_data = pd.get_dummies(titanic_data['Embarked']).astype(int)
titanic_data = pd.concat([titanic_data, sex_data, embarked_data], axis=1)
titanic_data.drop(['Name', 'PassengerId', 'Ticket', 'Sex', 'Embarked'], axis=1, inplace=True)
print(titanic_data.head(5))

# Train and evaluate Logistic Regression
X = titanic_data.drop("Survived", axis=1)
y = titanic_data["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("************Classification Report************\n")
print(classification_report(y_test, predictions))
print("***********Confusion Matrix***************")
print(confusion_matrix(y_test, predictions))


## `ex01c_logistic_diabetes.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

diabetes_data = pd.read_csv("../MLdatasets/diabetes.csv")
X = diabetes_data.drop("Outcome", axis=1)
y = diabetes_data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("************Classification Report************\n")
print(classification_report(y_test, predictions))
print("***********Confusion Matrix***************")
print(confusion_matrix(y_test, predictions))


## `ex02a_univariate_normal.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

x = np.linspace(-3, 5, 150)
params = [(0, 1, "$N(0,1)$"), (2, 3, "$N(2,3)$"), (0, 0.2, "$N(0,0.2)$")]

fig, ax = plt.subplots(figsize=(6, 4))

for mean, variance, label in params:
    pdf = norm.pdf(x, loc=mean, scale=np.sqrt(variance))
    ax.plot(x, pdf, label=label)

ax.set_title("Univariate Normal Distributions")
ax.set_xlabel("$x$")
ax.set_ylabel("density: $P(x)$")
ax.set_xlim((-3, 5))
ax.set_ylim((0, 1))
ax.legend()
plt.show()


## `ex02b_multivariate_normal.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

def generate_surface(mean, cov):
    x, y = np.linspace(-5, 5, 100), np.linspace(-5, 5, 100)
    x1, x2 = np.meshgrid(x, y)
    pos = np.dstack((x1, x2))
    pdf = multivariate_normal(mean=mean, cov=cov).pdf(pos)
    return x1, x2, pdf

distributions = [
    (np.array([0, 0]), np.eye(2)),
    (np.array([0, 0]), np.array([[1, -0.1], [-0.8, 1]]))
]

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
plt.suptitle("Bivariate Normal Distributions")
ax[0].set_title("Independent Variables")
ax[1].set_title("Correlated Variables")

for i, dist in enumerate(distributions):
    x1, x2, pdf = generate_surface(dist[0], dist[1])
    con = ax[i].contourf(x1, x2, pdf, 100, cmap='rainbow')
    ax[i].set_xlabel("$x_1$", fontsize=13)
    ax[i].set_ylabel("$x_2$", fontsize=13)
    ax[i].axis([-3.5, 3.5, -3.5, 3.5])
    c_bar = fig.colorbar(con, ax=ax[i])
    c_bar.ax.set_ylabel("$P(x_1, x_2)$", fontsize=13)

plt.show()


## `ex03a_pca_iris.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("../MLdatasets/Iris.csv")
X = data[data.columns[1:-1]]

scaled_X = StandardScaler().fit_transform(X)
reduced_X = PCA(n_components=2).fit_transform(scaled_X)

plt.figure(figsize=(7, 7))
sns.scatterplot(
    x=np.ndarray.ravel(reduced_X[:, :1]),
    y=np.ndarray.ravel(reduced_X[:, 1:]),
    hue=data['Species']
)
plt.title("Scatter Plot after reducing the dimension of feature vectors using PCA")
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid()
plt.show()


## `ex03b_pca_breast_cancer.py`

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data['data']

scaled_X = StandardScaler().fit_transform(X)
reduced_X = PCA(n_components=2).fit_transform(scaled_X)

plt.figure(figsize=(7, 7))
sns.scatterplot(
    x=np.ndarray.ravel(reduced_X[:, :1]),
    y=np.ndarray.ravel(reduced_X[:, 1:]),
    hue=data['target']
)
plt.title("Scatter Plot after reducing the dimension of feature vectors using PCA")
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid()
plt.show()


## `ex04a_kmeans_clustering.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans as sklKMeans
from sklearn.metrics import silhouette_score

data = pd.read_csv("../MLdatasets/Iris.csv")
X = data[["SepalLengthCm", "PetalLengthCm"]]

def kMeans(X, k):
    distortion = []
    for i in range(1, k + 1):
        model = sklKMeans(n_clusters=i, init='random', n_init='auto', max_iter=10).fit(X.values)
        distortion.append(model.inertia_)
    print(f"Silhoutte Score for {k} cluster is {silhouette_score(X, model.labels_)}")
    return model, distortion

k = 5
model, distortion = kMeans(X, k)

centroids = model.cluster_centers_
index = model.labels_

plt.figure(figsize=(14, 7))
plt.scatter(x=centroids[:, 0], y=centroids[:, 1], marker="+", c='b', linewidth=16)
sns.scatterplot(data=data, x="SepalLengthCm", y="PetalLengthCm", hue=index, palette="coolwarm")
plt.grid()
plt.savefig("K-Means_iris.png")
plt.show()

# Elbow Method for Optimal k
plt.figure(figsize=(7, 7))
sns.lineplot(x=np.arange(1, k + 1), y=np.array(distortion), marker="o", markersize=10)
plt.xticks(np.arange(1, k + 1))
plt.title("Elbow Method for Optimal k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Distortion")
plt.show()


## `ex04b_gmm_clustering.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

def gen_data(k=3, dim=2, points_per_cluster=2000, lim=[-10, 10]):
    x = []
    mean = np.random.rand(k, dim) * (lim[1] - lim[0]) + lim[0]
    for i in range(k):
        cov = np.random.rand(dim, dim + 10)
        cov = np.matmul(cov, cov.T)
        _x = np.random.multivariate_normal(mean[i], cov, points_per_cluster)
        x += list(_x)
    x = np.array(x)
    if (dim == 2):
        fig = plt.figure()
        ax = fig.gca()
        ax.scatter(x[:, 0], x[:, 1], s=3, alpha=0.4)
        ax.autoscale(enable=True)
    return x

data = gen_data()

gmm = GaussianMixture(n_components=3)
gmm.fit(data)

plt.figure(figsize=(8, 6))
plt.scatter(data[:, 0], data[:, 1], s=3)
for covar, mean in zip(gmm.covariances_, gmm.means_):
    v, w = np.linalg.eigh(covar)
    angle = np.arctan2(w[0][1], w[0][0])
    angle = 180 * angle / np.pi
    v = 2.0 * np.sqrt(2.0) * np.sqrt(v)
    ell = plt.matplotlib.patches.Ellipse(
        mean, v[0], v[1], 180 + angle, color='r', alpha=0.3
    )
    plt.gca().add_patch(ell)

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Gaussian Mixture Model')
plt.show()


## `ex05a_bpnn_classification.py`

In [ ]:
from sklearn.datasets import make_circles
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

n_samples = 1000
X, y = make_circles(n_samples, noise=0.03, random_state=42)

circles = pd.DataFrame({"X0": X[:, 0], "X1": X[:, 1], "label": y})
sns.scatterplot(data=circles, x="X0", y="X1", hue="label")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(2,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)
model.summary()

history = model.fit(X_train, y_train, epochs=10)
model.evaluate(X_test, y_test)

# Plot Model's Loss Curve
plt.figure()
plt.title("Model's Loss")
plt.plot(history.history['loss'])
plt.xlabel("No of Epoch")
plt.ylabel("Loss Value")
plt.show()

# Decision Boundary Helper Function
def plotq(model, X, y):
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    x_in = np.c_[xx.ravel(), yy.ravel()]
    y_pred = model.predict(x_in)
    if len(y_pred[0]) > 1:
        print("doing multiclass classification")
        y_pred = np.argmax(y_pred, axis=1).reshape(xx.shape)
    else:
        print("doing binary")
        y_pred = np.round(y_pred).reshape(xx.shape)
    plt.contourf(xx, yy, y_pred, cmap=plt.cm.RdYlBu, alpha=0.7)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.RdYlBu)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title('Training')
plotq(model, X_train, y_train)
plt.subplot(1, 2, 2)
plt.title('Testing')
plotq(model, X_test, y_test)
plt.show()


## `ex05b_svm_classification.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

iris = datasets.load_iris()
X = iris.data[:, :2]
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

svm_classifier = SVC(kernel='linear', C=1)
# Linear kernel with regularization parameter C=1
svm_classifier.fit(X_train, y_train)
y_pred = svm_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

report = classification_report(y_test, y_pred)
print("Classification Report:\n", report)

# Plot SVM Decision Boundary
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01), np.arange(y_min, y_max, 0.01))
Z = svm_classifier.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, cmap=plt.cm.RdYlBu, alpha=0.6)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu)
plt.xlabel('Sepal Length')
plt.ylabel('Sepal Width')
plt.title('SVM Classification')
plt.show()


## `ex06_decision_tree_random_forest.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load and Inspect Kyphosis Data
kyphosis_df = pd.read_csv('../MLdatasets/kyphosis.csv')
print(kyphosis_df.head())
kyphosis_df.info()

sns.pairplot(kyphosis_df, hue='Kyphosis')
plt.show()

# 2. Split Data
X = kyphosis_df.drop('Kyphosis', axis=1)
y = kyphosis_df['Kyphosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 3. Decision Tree Classifier
decision_tree_model = DecisionTreeClassifier()
decision_tree_model.fit(X_train, y_train)
decision_tree_y_pred = decision_tree_model.predict(X_test)

print("Classification Report for Decision Tree Model using Kyphosis Data:")
print(classification_report(y_test, decision_tree_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, decision_tree_y_pred))

# 4. Random Forest Classifier
random_forest_model = RandomForestClassifier()
random_forest_model.fit(X_train, y_train)
random_forest_y_pred = random_forest_model.predict(X_test)

print("Classification Report for Random Forest Model using Kyphosis Data:")
print(classification_report(y_test, random_forest_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, random_forest_y_pred))


## `ex07_cnn_mnist.py`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D,
    Dense, Dropout, Flatten,
    BatchNormalization
)
from tensorflow.keras.utils import to_categorical
import joblib

# 1. Load and Preprocess MNIST Data
train = pd.read_csv('../MLdatasets/mnist/train.csv')
test = pd.read_csv('../MLdatasets/mnist/test.csv')

X_train = train.drop(['label'], axis=1).values
y_train = train['label'].values
X_test = test.values

X_train = X_train.reshape(-1, 28, 28, 1) / 255
X_test = X_test.reshape(-1, 28, 28, 1) / 255
y_train = to_categorical(y_train)

# 2. Visualize Sample Digits
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap='binary')
    digit = y_train[i].argmax()
    ax.set(title=f"label: {digit}")
fig.tight_layout()
plt.show()

# 3. Define CNN Model Architecture
def get_model():
    model = Sequential()
    model.add(Conv2D(filters=64, kernel_size=(3, 3), activation="relu", input_shape=(28, 28, 1)))
    model.add(Conv2D(filters=64, kernel_size=(3, 3), activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(BatchNormalization())

    model.add(Conv2D(filters=128, kernel_size=(3, 3), activation="relu"))
    model.add(Conv2D(filters=128, kernel_size=(3, 3), activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(BatchNormalization())

    model.add(Conv2D(filters=256, kernel_size=(3, 3), activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(BatchNormalization())
    model.add(Dense(512, activation="relu"))
    model.add(Dense(10, activation="softmax"))

    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

tf.random.set_seed(0)
model = get_model()
print(model.summary())

# 4. Train and Save Model
history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=20,
    validation_split=0.2
)
tf.keras.models.save_model(model, "models/mnist_cnn.h5")
joblib.dump(history.history, "models/mnist_cnn.history")

# 5. Plot Loss and Accuracy Curves
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(history.history['loss'], color='b', label="Training loss")
ax[0].plot(history.history['val_loss'], color='r', label="validation loss")
ax[0].legend()

ax[1].plot(history.history['accuracy'], color='b', label="Training accuracy")
ax[1].plot(history.history['val_accuracy'], color='r', label="Validation accuracy")
ax[1].legend()
plt.show()

# 6. Predict on Test Images
y_pred = model.predict(X_test)
fig, axis = plt.subplots(4, 4, figsize=(8, 10))
for i, ax in enumerate(axis.flat):
    ax.imshow(X_test[i].squeeze(), cmap='binary')
    ax.set(title=f"Prediction: {y_pred[i].argmax()}")
    ax.axis("off")
fig.suptitle("Test Predictions")
fig.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()


## `ex08_lstm_sequence_prediction.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam, Adadelta
from sklearn.model_selection import train_test_split

# 1. Create Arithmetic Sequence
X = np.arange(1, 50)
Y = X * 15
print("Sequence:", X, Y, sep="\n")

X = X.reshape(-1, 1, 1)
Y = Y.reshape(-1, 1)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, shuffle=False)

# 2. Define LSTM Architecture
def get_model():
    model = Sequential()
    model.add(LSTM(50, activation='relu', input_shape=(1, 1)))
    model.add(Dense(1))
    model.compile(optimizer=Adadelta(learning_rate=0.03), loss='mse')
    return model

tf.random.set_seed(0)
model = get_model()
print(model.summary())

# 3. Train Model
model.fit(X_train, Y_train, epochs=2000, validation_split=0.2, batch_size=10)
tf.keras.models.save_model(model, "./models/seq_pred_model.h5")

# 4. Predict and Compare Actual vs Predicted
Y_pred = model.predict(X_test, verbose=0)
Y_diff = Y_test - Y_pred
df_results = pd.DataFrame(
    np.hstack([Y_test, Y_pred, Y_diff]),
    columns=["Actual", "Predicted", "Difference"]
)
print(df_results)


## `ex09_isolated_word_speech_recognition.py`

In [ ]:
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adadelta
from IPython.display import Audio, display

data_path = "./datasets/isolated_word_dataset"
labels = np.array(["bed", "cat", "happy"])
n_classes = labels.shape[0]
n_mfcc = 12  # no. of mfc coefficients
t = 30       # no. of time windows on which the mfc coefficients are computed
input_shape = (-1, n_mfcc, t, 1)

# 1. Extract MFCC Features from WAV Audio
def wav2mfcc(file, t, n_mfcc):
    data, sr = librosa.load(file, sr=None)
    mfcc = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=n_mfcc)
    if mfcc.shape[1] > t:
        mfcc = mfcc[:, :t]
    if mfcc.shape[1] < t:
        mfcc = np.pad(mfcc, pad_width=((0, 0), (0, t - mfcc.shape[1])))
    return mfcc

def load_speech_dataset_features(data_path, labels):
    X, y = [], []
    for i, label in enumerate(labels):
        file_name = f"{data_path}/{label}.npy"
        try:
            data = np.load(file_name)
        except FileNotFoundError:
            data = np.array([
                wav2mfcc(file, t=t, n_mfcc=n_mfcc)
                for file in librosa.util.find_files(f"{data_path}/{label}/")
            ])
            np.save(file_name, data)
        X.append(data)
        y.append(np.full((data.shape[0], 1), i))
    X = np.vstack(X).reshape(input_shape)
    y = np.vstack(y) == np.arange(n_classes)  # one-hot encoded
    return X, y

X, y = load_speech_dataset_features(data_path, labels)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 2. Build CNN Model for Spectrogram/MFCC Classification
def get_model():
    model = Sequential()
    model.add(Conv2D(32, kernel_size=(2, 2), activation='relu', input_shape=(n_mfcc, t, 1)))
    model.add(Conv2D(48, kernel_size=(3, 3), activation='relu'))
    model.add(Conv2D(64, kernel_size=(4, 4), activation='relu'))
    model.add(MaxPooling2D(pool_size=(4, 4)))
    model.add(Dropout(0.25))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.25))
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(n_classes, activation='softmax'))
    model.compile(
        loss=CategoricalCrossentropy(),
        optimizer=Adadelta(0.3),
        metrics=['accuracy']
    )
    return model

tf.random.set_seed(0)
model = get_model()
print(model.summary())

# 3. Train Model
model.fit(
    X_train, y_train,
    batch_size=50,
    epochs=50,
    verbose=True,
    validation_data=(X_test, y_test)
)
tf.keras.models.save_model(model, "./models/isolated_word_speech_recognition_model.h5")

# 4. Test on Recorded Audio Files
for label in labels:
    file_name = f"{data_path}/recorded_test_audios/{label}.wav"
    test_audio, sr = librosa.load(file_name, sr=None, mono=True)
    print(f"File Name: {label}.wav")
    display(Audio(test_audio, rate=sr))
    features = wav2mfcc(file_name, t, n_mfcc).reshape(input_shape)
    print("Predicted Output :", labels[np.argmax(model.predict(features))])
    print("-" * 40)


## `ex10_face_detection_opencv.py`

In [ ]:
import cv2

face_cascade = cv2.CascadeClassifier('models/haarcascade_frontalface_default.xml')
cap = cv2.VideoCapture('datasets/CERN_Higgs boson_EDIT.mp4')

while True:
    ok, img = cap.read()
    if not ok:
        print('Video Ending')
        cap.release()
        cv2.destroyAllWindows()
        break

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detect the faces
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)

    # Draw the rectangle around each face
    for (x, y, w, h) in faces:
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255))

    # Display
    cv2.imshow('img', img)

    # Press Key Q to quit and Close window
    if cv2.waitKey(1) & 0xFF == ord('q'):
        # Release the VideoCapture object
        cap.release()
        # Close all window
        cv2.destroyAllWindows()
        break


## `ex11_object_recognition_cifar10.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import Sequential
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, EarlyStopping
from tensorflow import random
import joblib
import pandas as pd

# 1. Load CIFAR-10 Dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
cifar_classes = np.array([
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'frog', 'dog', 'horse', 'ship', 'truck'
])

# 2. Preview 9 Training Samples
plt.figure(figsize=(10, 10))
plt.suptitle('Examples of training images and their labels')
for i in range(9):
    plt.subplot(331 + i, title=f'{y_train[i, 0]}-{cifar_classes[y_train[i, 0]]}')
    plt.imshow(X_train[i])
plt.show()

# 3. One-Hot Encode & Normalize Pixels
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

print(f"Shape of training data:\nX-train_shape={X_train.shape} , y-train_shape={y_train.shape}")
print(f"Shape of testing data:\nX-test_shape={X_test.shape} , y-test_shape={y_test.shape}")

# 4. Build CNN Model
def get_model():
    model = Sequential()
    model.add(Conv2D(25, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu', input_shape=(32, 32, 3)))
    model.add(Conv2D(50, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Conv2D(70, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Flatten())
    model.add(Dense(500, activation='relu'))
    model.add(Dropout(0.4))
    model.add(Dense(250, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(10, activation='softmax'))

    model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
    return model

random.set_seed(0)
model = get_model()
print(model.summary())

# 5. Train with ModelCheckpoint and EarlyStopping
checkpoint = ModelCheckpoint("./model/cifar10_epo_cnn.h5", monitor='val_accuracy', mode='max', save_best_only=True)
early_stop = EarlyStopping(monitor="val_accuracy", patience=5, mode="max")
call_backs = [early_stop, checkpoint]

history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,
    verbose=True,
    validation_split=0.2
)
model.save("./models/cifar10_epo_cnn.h5")
joblib.dump(history.history, "./models/cifar10_epo_cnn.history")

# 6. Plot Accuracy & Loss Curves
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(history.history['accuracy'], 'C0')
ax[0].plot(history.history['val_accuracy'], 'C1')
ax[0].set_title(label="Accuracy curves", fontsize=16)
ax[0].set_xlabel('Epochs', fontsize=14)
ax[0].set_ylabel('Accuracy', fontsize=14)
ax[0].grid()

ax[1].plot(history.history['loss'], 'C0')
ax[1].plot(history.history['val_loss'], 'C1')
ax[1].set_title(label="Loss curves", fontsize=16)
ax[1].set_xlabel('Epochs', fontsize=14)
ax[1].set_ylabel('Loss', fontsize=14)
ax[1].grid()
plt.legend(['Training', 'Validation'], fontsize=12)
plt.show()

# 7. Predict on First 15 Test Images & Count Classes
m = 15
y_pred_ = model.predict(X_test[:m])
y_pred_class = y_pred_.argmax(-1)
unique, counts = np.unique(y_pred_class, return_counts=True)

plt.figure(figsize=(10, 6))
plt.suptitle('Testing images and their labels')
for i in range(15):
    plt.subplot(3, 5, i + 1, title=f'{cifar_classes[y_pred_class[i]]}')
    plt.imshow(X_test[i])
plt.show()

freq = list(zip(cifar_classes[unique], counts))
print(pd.DataFrame(freq, columns=["class name", "count"]))
